In [1]:
!pip install unidic-lite
!pip install fugashi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=3dc3e2bc0fed4d7620362d3e2240016a5676f885c3e24a02e1a8d4fb96e78222
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built unidic-lite
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.9/697.9 kB 17.7 MB/s eta 0:00:00


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch import nn
from datasets import load_dataset
from transformers import AutoTokenizer

from collections import Counter
from tqdm.auto import tqdm, trange
import matplotlib.pyplot as plt
from time import time

In [3]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# torch.set_default_device(device)

In [4]:
ds = load_dataset("Verah/JParaCrawl-Filtered-English-Japanese-Parallel-Corpus");

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

1m_filtered.tsv.gz:   0%|          | 0.00/104M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

In [5]:
ds["train"]

Dataset({
    features: ['id', 'english', 'japanese', 'model1_accepted', 'model2_accepted'],
    num_rows: 1000000
})

In [6]:
jpn_tokenizer = AutoTokenizer.from_pretrained("cl-tohoku/bert-base-japanese")

tokenizer_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [7]:
jpn_tokenizer.vocab_size

32000

In [8]:
# jpn_smaple = cleaned_ds[0]["japanese"]
# jpn_tokenizer.encode(jpn_smaple)

In [9]:
class Eng2JpnDataset(Dataset):
    def __init__(self, dataset, japanese_tokenizer, eager_tokenizer=False, max_length=512, mode="train"):
        if mode == "train":
            self.japanese_tokenizer = japanese_tokenizer
            self.eager_tokenizer = eager_tokenizer
            self.raw_data = dataset
            self.max_length = max_length
            self.eng_vocab = self._build_vocab(self.raw_data, "english")

            self.idx2word, self.word2idx = self._build_dict(self.eng_vocab)

            if self.eager_tokenizer:
                self.english_tokenized = []
                self.japanese_tokenized = []
                for english, japanese in zip(self.raw_data["english"], self.raw_data["japanese"]):
                    self.english_tokenized.append(self._eng_tokenize(english))
                    self.japanese_tokenized.append(self._jpn_tokenize(japanese))


    def _build_vocab(self, dataset, language, min_freq=10):
        vocab = Counter()
        for sentence in dataset[language]:
            words = [word.lower() for word in sentence.split()]
            vocab.update(words)

        vocab = [word for word, count in vocab.items() if count >= min_freq]
        return sorted(vocab)


    def _build_dict(self, vocab):
        word2idx = {word : idx+4 for idx, word in enumerate(vocab)}
        idx2word = {idx+4 : word for idx, word in enumerate(vocab)}

        word2idx["<PAD>"] = 0
        word2idx["<UNK>"] = 1
        word2idx["<BOS>"] = 2
        word2idx["<EOS>"] = 3

        idx2word[0] = "<PAD>"
        idx2word[1] = "<UNK>"
        idx2word[2] = "<BOS>"
        idx2word[3] = "<EOS>"

        return idx2word, word2idx


    def _eng_tokenize(self, eng_sentence):
        tokenized = []
        for word in eng_sentence.split():
            word = word.lower()
            tokenized.append(self.word2idx.get(word, self.word2idx["<UNK>"]))
        return tokenized

    def _jpn_tokenize(self, jpn_sentence):
        return self.japanese_tokenizer.encode(jpn_sentence, max_length=self.max_length, truncation=True)


    def __len__(self):
        return len(self.raw_data["english"])


    def __getitem__(self, index):
        if self.eager_tokenizer:
            return self.english_tokenized[index], self.japanese_tokenized[index]

        else:
            eng, jpn = self._eng_tokenize(self.raw_data["english"][index]), self._jpn_tokenize(self.raw_data["japanese"][index])
            return torch.tensor(eng[:self.max_length], dtype=torch.long), torch.tensor(jpn[:self.max_length], dtype=torch.long)


In [10]:
cleaned_ds = ds["train"].filter(lambda x : x["model1_accepted"] == 1 or x["model2_accepted"] == 1)

Filter:   0%|          | 0/1000000 [00:00<?, ? examples/s]

In [11]:
len(cleaned_ds)

613669

In [12]:
train_val_test = cleaned_ds.train_test_split(test_size=0.99, seed=42) # for efficiency resons
train_val = train_val_test["train"].train_test_split(test_size=0.3, seed=42)

# test_dataset = train_val_test["test"]
val_dataset = train_val["test"]
train_dataset = train_val["train"]

In [13]:
start = time()
train_custom_dataset = Eng2JpnDataset(train_dataset, jpn_tokenizer)
val_custom_dataset = Eng2JpnDataset(val_dataset, jpn_tokenizer)
# test_custom_dataset = Eng2JpnDataset(test_dataset, jpn_tokenizer)
f"creating datasets took : {(time() - start):.2f}s"

'creating datasets took : 0.95s'

In [14]:
def collate_pad(batch):
    eng_batch, jpn_batch = zip(*batch)
    eng_padded = pad_sequence(eng_batch, batch_first=True)
    jpn_padded= pad_sequence(jpn_batch, batch_first=True)

    eng_lengths = torch.tensor([len(seq) for seq in eng_batch], device="cpu")
    jpn_lengths = torch.tensor([len(seq) for seq in jpn_batch], device="cpu")
    return (eng_padded, jpn_padded), (eng_lengths, jpn_lengths)


BATCH_SIZE = 8
pin_memory = True
train_loader = DataLoader(train_custom_dataset, BATCH_SIZE, True, collate_fn=collate_pad, pin_memory=pin_memory)
val_loader = DataLoader(val_custom_dataset, BATCH_SIZE, False, collate_fn=collate_pad, pin_memory=pin_memory)
# test_loader = DataLoader(test_custom_dataset, BATCH_SIZE, False, collate_fn=collate_pad, pin_memory=pin_memory)

In [15]:
class Encoder(nn.Module):
    def __init__(self, vocab_dim, embed_dim=64, hidden_dim=64, num_layers=2, padding_idx=0):
        super().__init__()
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_dim, embed_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)


    def forward(self, src, src_lengths):
        embed = self.embedding(src)
        packed = pack_padded_sequence(embed, src_lengths, True, False)
        packed_output, (h_n, c_n) = self.lstm(packed) # n_layers*bidirectional, Batch_size, Hidden_dim
        output, _ = pad_packed_sequence(packed_output, batch_first=True)

        ########################
        last_h = h_n.view(self.num_layers, 2, h_n.size(1), h_n.size(2)) #num_layers, 2, batchsize, hidden_dim
        last_c = c_n.view(self.num_layers, 2, h_n.size(1), h_n.size(2))
        o_h = torch.cat([last_h[:, 0], last_h[:, 1]], dim=-1) #L, B, 2*H
        o_c = torch.cat([last_c[:, 0], last_c[:, 1]], dim=-1)
        return output, (o_h, o_c)


class Decoder(nn.Module):
    def __init__(self, vocab_dim, embed_dim=64, hidden_dim=64, num_layers=2, padding_idx=0):
        super().__init__()
        self.hidden_dim = hidden_dim * 2
        self.embedding = nn.Embedding(vocab_dim, embed_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embed_dim, self.hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(self.hidden_dim, vocab_dim)


    def forward(self, encoder_o, bos=2, eos=3, max_len=512):

        h, c = encoder_o
        assert h.size(-1) == self.hidden_dim or c.size(-1) == self.hidden_dim, "encoder hidden state shape is not the same as decoder hidden state"
        batch_size = h.size(1)
        # index_token_generated = []
        all_logits = []
        input_tokens = torch.full((batch_size,), bos, device=h.device) #type & device
        embed = self.embedding(input_tokens).unsqueeze(1)
        finished = torch.zeros(batch_size, dtype=torch.bool)

        for n_token_generated in trange(max_len, leave=False, desc="generating translation..."):
            output, (h, c) = self.lstm(embed, (h, c))
            next_logit = self.fc(h[-1])
            all_logits.append(next_logit)
            next_token = next_logit.argmax(dim=-1)
            # index_token_generated.append(next_token)
            embed = self.embedding(next_token).unsqueeze(1)
            mask = (next_token == eos)
            finished[mask] = 1
            if finished.all():
                print(f"all sentences finished in {n_token_generated}")
                break

        return torch.stack(all_logits, dim=1)


class Seq2Seq(nn.Module):
    def __init__(self, original_vocab_dim, translated_vocab_dim):
        super().__init__()
        self.encoder = Encoder(original_vocab_dim)
        self.decoder = Decoder(translated_vocab_dim)


    def forward(self, text, src_lenghts):
        _, (o_h, o_c) = self.encoder(text, src_lenghts)
        translation = self.decoder((o_h, o_c))
        return translation

In [16]:
def train_and_evaluate_one_epoch(model, optimizer, criterion, device, train_loader, val_loader, grad_scaler, scheduler): #assumes model on device

    iter_val = iter(val_loader)
    iteration = 0
    for (original_text, translation), (eng_lengths, _) in tqdm(train_loader, leave=True):
        start = time()
        model.train()
        original_text, translation = original_text.to(device, non_blocking=True), translation.to(device, non_blocking=True)

        optimizer.zero_grad()
        pred_token = model(original_text, eng_lengths)
        with torch.amp.autocast(device_type=device.type):

            padded = pad_sequence(translation, batch_first=True, padding_value=0)

            # truncate to fixed length
            fixed_len = pred_token.size(1)
            if padded.size(1) > fixed_len:
                padded = padded[:, :fixed_len]
            else:
                # optionally pad to fixed_len if needed
                pad_size = fixed_len - padded.size(1)
                padded = torch.cat([padded, torch.zeros(padded.size(0), pad_size, dtype=torch.long, device=device)], dim=1)
            B, S, V = pred_token.shape

            train_loss = criterion(pred_token.view(-1, V), padded.view(-1))
        grad_scaler.scale(train_loss).backward()
        grad_scaler.unscale_(optimizer)
        optimizer.step()
        grad_scaler.update()
        scheduler.step()

        try:
            (x_val, y_val), (eng_lengths, _) = next(iter_val)
        except StopIteration:
            iter_val = iter(val_loader)
            (x_val, y_val), (eng_lengths, _) = next(iter_val)


        with torch.inference_mode():
            model.eval()
            x_val, y_val = x_val.to(device, non_blocking=True), y_val.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type):
                pred_val = model(x_val, eng_lengths)

                padded = pad_sequence(y_val, batch_first=True, padding_value=0)

                # truncate to fixed length
                fixed_len = pred_val.size(1)
                if padded.size(1) > fixed_len:
                    padded = padded[:, :fixed_len]
                else:
                    # optionally pad to fixed_len if needed
                    pad_size = fixed_len - padded.size(1)
                    padded = torch.cat([padded, torch.zeros(padded.size(0), pad_size, dtype=torch.long, device=device)], dim=1)
                B, S, V = pred_val.shape

                val_loss = criterion(pred_val.view(-1, V), padded.view(-1))
            iteration += 1
            if iteration % 32:
                print(f"validation loss : {val_loss.item():.4f} | train loss : {train_loss.item():.4f} | took {(time() - start):.2f}s")
    return model

In [17]:
# config
N_EPOCHS = 1
lr = 1e-2
input_vocab_dim = len(train_loader.dataset.eng_vocab)
output_vocab_dim = train_loader.dataset.japanese_tokenizer.vocab_size
loss_function = nn.CrossEntropyLoss() #we should ignore padding index
model = Seq2Seq(input_vocab_dim, output_vocab_dim)
model = model.to(device, non_blocking=True)
model = torch.compile(model.to(device, non_blocking=True))
opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
grad_scaler = torch.amp.GradScaler()
scheduler = CosineAnnealingLR(opt, T_max=len(train_loader), eta_min=1e-6)

In [18]:
import torch, gc

# delete any large objects you don't need
# del model, opt, train_loader, val_loader
gc.collect()
torch.cuda.empty_cache()

gc.collect()                 # clear Python memory
torch.cuda.empty_cache()     # free cached GPU memory


In [ ]:
for epoch in trange(N_EPOCHS):
    train_losses, val_losses = [], []
    model_trained = train_and_evaluate_one_epoch(model, opt, loss_function, device, train_loader, val_loader, grad_scaler, scheduler)
    train_losses.append(t_loss)
    val_losses.append(v_loss)


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/537 [00:00<?, ?it/s]

generating translation...:   0%|          | 0/512 [00:00<?, ?it/s]

generating translation...:   0%|          | 0/512 [00:00<?, ?it/s]

validation loss : 10.3062 | train loss : 10.5077 | took 26.05s
